# Random circuits II: non-Markovian U(1)

**Manuscript map.** Sec. III G 1, Eq. (78).
The channel-mode audit below uses Eq. (3.10) of
[Marvian and Spekkens](https://arxiv.org/pdf/1312.0680).
The SU(2) non-Markovian companion
[`asymm_ex5.ipynb`](asymm_ex5.ipynb) applies the same
Eq. (3.10) as a contraction bound on the rank-1
irreducible tensor.

Here the environment is a finite quantum memory. It starts in
$|0\rangle^{\otimes N_e}$ and is never reset, so information and
asymmetry can return to the system.

**Computation cost.** Producing the stored data required
3022.5 seconds (50.4 minutes) on a 10-core Apple M4 MacBook Air
with 16 GB of memory, using 10 concurrent circuit workers. The
calculation contains 100 circuits, 1000 Floquet layers, five
tilts, and a joint Hilbert-space dimension of
$2^{20}=1{,}048{,}576$, for 500,500 exact reduced-state samples.
The occupied-U(1)-sector engine performs no interpolation,
curve fitting, or precision reduction.

The main dataset is the complete production ensemble:
100 archived circuits, 1000 Floquet layers, and the manuscript
Hilbert-space size $N_s=8,N_e=12$. Every gate row comes from
[`alessum/mpemba_circuits`](https://github.com/alessum/mpemba_circuits)
at the pinned commit recorded below.

A smaller $N_s=4,N_e=8$ file is retained only for the inexpensive
explicit-channel audit of Marvian--Spekkens Eq. (3.10). It is not
used for the ensemble averages or crossing times.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg as la

_HERE = Path.cwd().resolve()
for _candidate in [_HERE, *_HERE.parents]:
    if (_candidate / "tools" / "notebook_utils.py").exists():
        _TOOLS_DIR = _candidate / "tools"
        break
else:
    raise RuntimeError(
        "Could not locate tools/notebook_utils.py from cwd or its parents."
    )
if str(_TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(_TOOLS_DIR))
_REPO_ROOT = _TOOLS_DIR.parent

import circuit_data as cd
import notebook_utils as nu

np.set_printoptions(precision=5, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "figure.figsize": (7.2, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "font.size": 11,
})

BLUE = "#394F87"
ORANGE = "#E8792E"
GOLD = "#F4C95D"
PALETTE_3 = [BLUE, ORANGE, GOLD]
PALETTE_5 = plt.cm.Oranges(np.linspace(0.35, 0.95, 5))


def load_npz(path):
    with np.load(path, allow_pickle=False) as stored:
        return {name: stored[name] for name in stored.files}


## 1. Load the raw arrays and audit their provenance

Both files store every monotone value and every gate parameter;
the notebook performs the reductions itself. The exact-size file
contains all archived circuit rows $0,\ldots,99$ and every time
sample $t=0,\ldots,1000$.

Set `REGENERATE_SCALED=True` to rerun the local 24-circuit
ensemble used by the channel audit. Recreating the paper-scale
file from its upstream parameter table is an explicit command
shown below the audit.


In [ ]:
REFERENCE_PATH = _REPO_ROOT / "data" / "circuit_examples" / "u1_nonmarkovian_reference.npz"
SCALED_PATH = _REPO_ROOT / "data" / "circuit_examples" / "u1_nonmarkovian.npz"
REGENERATE_SCALED = False

if REGENERATE_SCALED:
    generated = cd.generate_u1_nonmarkovian()
    cd.save_dataset(SCALED_PATH, generated)

for path in (REFERENCE_PATH, SCALED_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} is missing; see "
            "data/circuit_examples/README.md."
        )

reference = load_npz(REFERENCE_PATH)
scaled = load_npz(SCALED_PATH)

reference_asymmetry = reference["asymmetry"]
reference_times = reference["times"]
# All ensemble statistics below use the full exact-size run.
asymmetry = reference_asymmetry
times = reference_times
theta_over_pi = reference["theta_over_pi"]
labels = [fr"$\theta={theta:.2f}\pi$" for theta in theta_over_pi]
# Dark orange denotes the smallest tilt and light orange
# denotes the largest one.
theta_colors = PALETTE_5[::-1]

print(
    "reference source:",
    reference["source_repository"].item(),
)
print(
    "commit:", reference["source_commit"].item()
)
print(
    "parameter SHA-256:",
    reference["source_parameter_sha256"].item(),
)
print(
    "reference circuits:",
    f"{reference['source_indices'][0]}..."
    f"{reference['source_indices'][-1]}",
)
print(
    "exact-size raw array:", reference_asymmetry.shape,
    f"(Ns={reference['n_system'].item()}, "
    f"Ne={reference['n_environment'].item()})",
)
print(
    "scaled audit array:", scaled["asymmetry"].shape,
    f"(Ns={scaled['n_system'].item()}, "
    f"Ne={scaled['n_environment'].item()})",
)
print(
    "simulation engine:",
    reference["simulation_engine"].item(),
)
print(
    "simulation workers:",
    reference["simulation_workers"].item(),
)

assert asymmetry.shape == (
    100, 5, 1001
)
assert scaled["asymmetry"].shape == (
    int(scaled["n_realizations"]),
    5,
    len(scaled["times"]),
)
assert np.isfinite(asymmetry).all()
assert np.isfinite(scaled["asymmetry"]).all()
assert np.array_equal(times, np.arange(1001))
assert np.array_equal(
    reference["source_indices"], np.arange(100)
)
assert int(reference["n_system"]) == 8
assert int(reference["n_environment"]) == 12
assert reference["source_commit"].item() == (
    "5042f3600a5b14c93b515e0bd0dab0e8fa4d5509"
)


In [ ]:
# Reproduction command for the exact-size reference file:
#
# python tools/generate_circuit_data.py \
#   --only u1-reference \
#   --paper-scale \
#   --reference-checkout /path/to/mpemba_circuits
#
# The generator verifies the upstream parameter file's SHA-256
# and uses the exact occupied-sector engine.


## 2. Marvian--Spekkens mode transfer: Eq. (3.10)

Let $\{T_m^{(\mu,\alpha)}\}$ and
$\{S_m^{(\mu,\beta)}\}$ be Hilbert--Schmidt-orthonormal
irreducible tensor-operator bases at the input and output.
Marvian and Spekkens' Eq. (3.10) is

$$
\mathcal E(X)
=
\sum_{\mu,m,\alpha}
\operatorname{Tr}\!\left[
    T_m^{(\mu,\alpha)\dagger}X
\right]
\sum_\beta
c_{\beta\alpha}^{(\mu)}
S_m^{(\mu,\beta)},
\qquad
c_{\beta\alpha}^{(\mu)}
=
\operatorname{Tr}\!\left[
    S_m^{(\mu,\beta)\dagger}
    \mathcal E(T_m^{(\mu,\alpha)})
\right].
$$

For U(1), every irrep is one-dimensional, so $m$ is
trivial and $\mu=\omega$ is the charge difference. We use the
normalized matrix units
$T^{(\omega,\alpha)}=S^{(\omega,\alpha)}
=|r\rangle\langle s|$, with
$\omega=q_r-q_s$. The multiplicity label $\alpha=(r,s)$
distinguishes all matrix units with the same $\omega$.
Consequently, each matrix $c^{(\omega)}$ is exactly one
charge-difference block of the channel superoperator.

The calculation below constructs the one-layer reduced channel
for the first stored $N_s=4,N_e=8$ circuit. Its environment is
initialized in the U(1)-invariant density operator
$|0\cdots0\rangle\langle0\cdots0|$. It then verifies Eq. (3.10)
in two independent ways: cross-mode entries vanish, and the
sum over the $c^{(\omega)}$ blocks reconstructs
$\mathcal E(X)$ for a generic operator.

Eq. (3.10) is a transfer formula, not a contraction bound. The
trace-norm bound on a multiplicity-free coefficient appears
later as Eqs. (3.12)--(3.13) of the same reference.


In [ ]:
eq310_n_system = int(scaled["n_system"])
eq310_n_environment = int(scaled["n_environment"])
eq310_n_total = (
    eq310_n_system + eq310_n_environment
)
eq310_d_system = 2**eq310_n_system
eq310_d_environment = 2**eq310_n_environment

eq310_environment = np.zeros(
    eq310_d_environment, dtype=complex
)
eq310_environment[0] = 1
# Column j is |0...0>_e tensor |j>_s.
eq310_joint_basis = np.kron(
    eq310_environment[:, None],
    np.eye(eq310_d_system, dtype=complex),
)
eq310_parameters = scaled["gate_parameters"][0]
eq310_gates = [
    cd.fn.gen_u1(row.tolist())
    for row in eq310_parameters
]
eq310_ordering = cd.fn.gen_gates_order(
    eq310_n_total, geometry="brickwork"
)
eq310_masks = cd.fn.load_mask_memory(eq310_n_total)
eq310_evolved_basis = cd._apply_layer_batch(
    eq310_joint_basis,
    eq310_gates,
    eq310_ordering,
    eq310_masks,
)

# A[i, e, j] is the amplitude for system output i and
# environment output e given system input j.
eq310_amplitudes = eq310_evolved_basis.reshape(
    eq310_d_environment,
    eq310_d_system,
    eq310_d_system,
).transpose(1, 0, 2)
eq310_channel_tensor = np.einsum(
    "iej,kel->ikjl",
    eq310_amplitudes,
    eq310_amplitudes.conj(),
    optimize=True,
)
eq310_channel = eq310_channel_tensor.reshape(
    eq310_d_system**2,
    eq310_d_system**2,
    order="F",
)

eq310_blocks = nu.channel_charge_blocks(
    eq310_channel, eq310_n_system
)
eq310_same_mode_channel = np.zeros_like(
    eq310_channel
)
print("omega   dim c^(omega)")
for omega, (transfer, indices) in eq310_blocks.items():
    eq310_same_mode_channel[np.ix_(indices, indices)] = (
        transfer
    )
    print(f"{omega:>3d}     {len(indices):>3d}")

eq310_cross_mode_error = la.norm(
    eq310_channel - eq310_same_mode_channel
)

eq310_rng = np.random.default_rng(310)
eq310_test_operator = (
    eq310_rng.normal(
        size=(eq310_d_system, eq310_d_system)
    )
    + 1j
    * eq310_rng.normal(
        size=(eq310_d_system, eq310_d_system)
    )
)
eq310_test_operator /= la.norm(eq310_test_operator)
eq310_input_coefficients = nu.vec(
    eq310_test_operator
)
eq310_output_coefficients = np.zeros_like(
    eq310_input_coefficients
)
for transfer, indices in eq310_blocks.values():
    eq310_output_coefficients[indices] = (
        transfer
        @ eq310_input_coefficients[indices]
    )
eq310_direct_output = nu.unvec(
    eq310_channel @ eq310_input_coefficients,
    eq310_d_system,
)
eq310_reconstructed_output = nu.unvec(
    eq310_output_coefficients,
    eq310_d_system,
)
eq310_reconstruction_error = la.norm(
    eq310_direct_output
    - eq310_reconstructed_output
)

eq310_weights = nu.hamming_weights(
    eq310_n_system
)
eq310_phase = 0.731
eq310_rotation = np.diag(
    np.exp(-1j * eq310_phase * eq310_weights)
)

def eq310_action(operator):
    return nu.unvec(
        eq310_channel @ nu.vec(operator),
        eq310_d_system,
    )


eq310_covariance_error = la.norm(
    eq310_action(
        eq310_rotation
        @ eq310_test_operator
        @ eq310_rotation.conj().T
    )
    - eq310_rotation
    @ eq310_action(eq310_test_operator)
    @ eq310_rotation.conj().T
)
eq310_identity_vector = nu.vec(
    np.eye(eq310_d_system)
)
eq310_trace_preservation_error = la.norm(
    eq310_identity_vector.conj() @ eq310_channel
    - eq310_identity_vector.conj()
)

print(
    "cross-mode Frobenius error:",
    eq310_cross_mode_error,
)
print(
    "Eq. (3.10) reconstruction error:",
    eq310_reconstruction_error,
)
print(
    "direct U(1)-covariance error:",
    eq310_covariance_error,
)
print(
    "trace-preservation error:",
    eq310_trace_preservation_error,
)

assert eq310_cross_mode_error < 1e-12
assert eq310_reconstruction_error < 1e-12
assert eq310_covariance_error < 1e-12
assert eq310_trace_preservation_error < 1e-12


## 3. What is evolved?

The system state is the homogeneous product tilt

$$
|\varphi(\theta)\rangle =
\bigotimes_{n=1}^{N_s} e^{-i\theta s_y^{(n)}}|0\rangle_n.
$$

Each realization samples one U(1)-symmetric brickwork layer from
Eqs. (71)-(72) and repeatedly applies it to the *joint* pure state.
The exact engine evolves the nine occupied total-charge sectors
once and reuses them for all five tilts. At every time it
reconstructs the complete reduced density matrices from shared
environment-weight Gram products. Thus the plotted values are
$S(\mathcal G_{\rm U(1)}[\rho_s])-S(\rho_s)$ evaluated on actual
reduced states, without interpolation or fitted curves.

For the exact-size file, the gate parameters are not resampled:
they are rows of `data/U1_rnd_parameters.npy` in the linked
reference repository. The sector implementation agrees with the
dense `functions.apply_U` convention and with the former
exact-size reference trajectories to floating-point precision.


## 4. Raw manuscript-size trajectories

The complete file contains 100 independent $2^{20}$-dimensional
circuit evolutions. Six evenly spaced circuit rows are shown
below so the raw variability remains visible without creating
an unreadable 100-panel figure. No fitted or illustrative curve
enters these panels.


In [ ]:
shown_reference_rows = np.linspace(
    0,
    reference_asymmetry.shape[0] - 1,
    6,
    dtype=int,
)
fig, axes = plt.subplots(
    2,
    3,
    figsize=(11, 6.5),
    sharex=True,
    sharey=True,
)
for realization, ax in zip(
    shown_reference_rows, axes.ravel()
):
    for state_index, (label, color) in enumerate(
        zip(labels, theta_colors)
    ):
        ax.plot(
            reference_times[1:],
            reference_asymmetry[
                realization, state_index, 1:
            ],
            color=color,
            lw=1.2,
            label=label,
        )
    ax.set(
        xscale="log",
        xlabel="Floquet layer",
        title=(
            "reference circuit "
            f"{reference['source_indices'][realization]}"
        ),
    )
axes[0, 0].set_ylabel(r"$M_{\rm U(1)}[\rho_s(t)]$")
axes[1, 0].set_ylabel(r"$M_{\rm U(1)}[\rho_s(t)]$")
axes[0, -1].legend(fontsize=7)
fig.tight_layout()
plt.show()


## 5. Ensemble-averaged symmetry restoration

The panel below averages each tilt over all 100 manuscript-size
circuits. Logarithmic axes expose the complete time range and
the nonexponential, nonmonotonic relaxation caused by the finite
unreinitialized environment. The crossing times are calculated
from the same mean curves and printed numerically for use in the
uncertainty analysis that follows.


In [ ]:
mean_curves = asymmetry.mean(axis=0)
paired_differences = (
    asymmetry[:, 1:, :]
    - asymmetry[:, :1, :]
)
mean_differences = paired_differences.mean(axis=0)
crossing_times = np.array([
    nu.crossing_time(
        times,
        mean_curves[state_index],
        mean_curves[0],
    )
    for state_index in range(1, len(theta_over_pi))
])

print("tilt        D_theta(0)    mean crossing")
for state_index, tau in enumerate(
    crossing_times, start=1
):
    print(
        f"{theta_over_pi[state_index]:.2f}pi"
        f"       {mean_differences[state_index - 1, 0]: .6f}"
        f"       t={tau:.2f}"
    )

selected_crossing = crossing_times[-1]

fig, ax = plt.subplots(figsize=(7.6, 4.5))
for curve, theta, color in zip(
    mean_curves,
    theta_over_pi,
    theta_colors,
):
    ax.plot(
        times[1:],
        curve[1:],
        color=color,
        lw=2,
        label=fr"${theta:.2f}\,\pi$",
    )
ax.set(
    xscale="log",
    yscale="log",
    xlim=(1, 1000),
    xlabel=r"$t$",
    ylabel=(
        r"$M[\rho_\theta(t)]"
        r"=S(\rho_\theta(t)\Vert"
        r"\mathcal{G}[\rho_\theta(t)])$"
    ),
)
ax.text(
    0.52,
    0.96,
    r"$\mathrm{U(1)}$" + "\nnon-Markovian",
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=9,
)
ax.legend(
    title=r"Values of $\theta$:",
    fontsize=8,
    title_fontsize=9,
    loc="upper right",
)
fig.tight_layout()
plt.show()

assert np.isfinite(crossing_times).all()
assert mean_differences[-1, 0] > 0
assert mean_differences[-1, -1] < 0


## 6. Crossing uncertainty and information backflow

The two diagnostics below answer different questions.

Define the paired mean difference

$$
D_\theta(t)
=
\left\langle
    M_\theta(t)-M_{0.30\pi}(t)
\right\rangle_{\rm circuits}.
$$

Its positive-to-negative zero crossing is the ordering reversal.
Pairing the two tilts circuit by circuit removes much of the
between-circuit variation.

1. The left panel attaches a paired 95% standard-error band to
   $D_{0.50\pi}(t)$. Its zero crossing is the ordering reversal
   of the two ensemble means. The band quantifies the
   realization uncertainty of the 100-circuit paper-scale
   average.
2. The right panel tests non-Markovian backflow. For every raw
   circuit and tilt we form
   $\Delta M(t)=M(t+1)-M(t)$. It reports the fraction of those
   500 raw trajectories for which $\Delta M(t)>0$.

A crossing does not by itself imply backflow: it compares two
initial states at the same time. Conversely, a positive
increment is a revival along one trajectory. Such a revival is
forbidden under repeated application of a fixed covariant
reset channel, but is allowed here because the unreinitialized
environment retains memory.


In [ ]:
increments = np.diff(asymmetry, axis=-1)
revival_fraction = (increments > 1e-10).mean(axis=(0, 1))
revival_window = 11
revival_padding = revival_window // 2
revival_fraction_smoothed = np.convolve(
    np.pad(
        revival_fraction,
        revival_padding,
        mode="edge",
    ),
    np.ones(revival_window) / revival_window,
    mode="valid",
)
selected_raw_difference = paired_differences[:, -1, :]
selected_mean_difference = (
    selected_raw_difference.mean(axis=0)
)
selected_sem_difference = (
    selected_raw_difference.std(axis=0, ddof=1)
    / np.sqrt(selected_raw_difference.shape[0])
)

print(
    "largest positive one-step change in the raw data:",
    increments.max(),
)
print(
    "fraction of all raw steps with a revival:",
    (increments > 1e-10).mean(),
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].plot(
    times[1:],
    selected_mean_difference[1:],
    color=theta_colors[-1],
    lw=2,
)
axes[0].fill_between(
    times[1:],
    (
        selected_mean_difference
        - 1.96 * selected_sem_difference
    )[1:],
    (
        selected_mean_difference
        + 1.96 * selected_sem_difference
    )[1:],
    color=theta_colors[-1],
    alpha=0.16,
    linewidth=0,
    label="paired 95% standard-error band",
)
axes[0].axhline(0, color="0.6", lw=1)
axes[0].axvline(
    selected_crossing,
    color="0.4",
    ls=":",
    lw=1,
)
axes[0].set(
    xscale="log",
    xlabel="Floquet layer",
    ylabel=r"$D_{0.50\pi}(t)$",
    title="uncertainty of the selected mean crossing",
)
axes[0].legend(fontsize=8)

axes[1].plot(
    times[1:],
    revival_fraction,
    color=ORANGE,
    alpha=0.20,
    lw=0.8,
    label="raw fraction at each layer",
)
axes[1].plot(
    times[1:],
    revival_fraction_smoothed,
    color=ORANGE,
    lw=2,
    label=f"{revival_window}-layer moving average",
)
axes[1].set(
    xscale="log",
    ylim=(-0.02, 1.02),
    xlabel="Floquet layer",
    ylabel=r"fraction with $\Delta M(t)>0$",
    title=(
        f"backflow among {asymmetry.shape[0]} "
        "circuits x 5 tilts"
    ),
)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

assert increments.max() > 0
assert np.any(revival_fraction > 0)


## Takeaway

The complete 100-circuit manuscript-size run exposes the ordering
reversal directly through the paired differences $D_\theta(t)$.
The highlighted mean crossing is visible, while its
standard-error band records the realization uncertainty of the
full archived ensemble. The separate
positive-increment statistic demonstrates non-Markovian
revivals: unlike a reset channel, a finite environment can return
asymmetry to the system.

Marvian--Spekkens Eq. (3.10) shows simultaneously that this
return cannot come from mixing different charge-difference
modes: each reduced channel from the initial time remains
block diagonal in $\omega$. Memory instead makes the transfer
matrices $c_t^{(\omega)}$ depend on the elapsed joint
evolution; there need not be a state-independent CP map from
one reduced-time snapshot to the next.

The full dataset can be regenerated with:
`tools/generate_circuit_data.py --only u1-reference
--paper-scale --reference-checkout ...`. The exact occupied-sector
engine makes this a reproducible local calculation; the stored
result contains every raw trajectory and time sample.
